### RapidFire AI Tutorial Use Case: SFT for Customer Support Q&A Chatbot

##### Note: This large FSDP recipe originally targeted 8× A10 GPUs. The notebook now sets `NUM_GPUS` from `torch.cuda.device_count()` so it matches your machine (you may have fewer GPUs; the run still schedules with `num_gpus=NUM_GPUS`). Data is heavily downsampled for demo purposes.

In [1]:
import torch
from rapidfireai import Experiment
from rapidfireai.automl import List, RFGridSearch, RFModelConfig, RFLoraConfig, RFSFTConfig

NUM_GPUS = max(1, int(torch.cuda.device_count()))
print(f"NUM_GPUS={NUM_GPUS} (torch.cuda.device_count()={torch.cuda.device_count()})")

NUM_GPUS=1 (torch.cuda.device_count()=1)


### Load Dataset and Specify Train and Eval Partitions

In [2]:
from datasets import load_dataset

dataset=load_dataset("bitext/Bitext-customer-support-llm-chatbot-training-dataset")

train_dataset=dataset["train"].select(range(320))
eval_dataset=dataset["train"].select(range(320,336))
train_dataset=train_dataset.shuffle(seed=42)
eval_dataset=eval_dataset.shuffle(seed=42)

### Define Data Processing Function

In [3]:
def sample_formatting_function(row):
    """Function to preprocess each example from dataset"""
    # Special tokens for formatting
    SYSTEM_PROMPT = "You are a helpful and friendly customer support assistant. Please answer the user's query to the best of your ability."
    return {
        "prompt": [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": row["instruction"]},
            
        ],
        "completion": [
            {"role": "assistant", "content": row["response"]}
        ]
    }

### Initialize Experiment

In [4]:
# Every experiment instance must be uniquely named
experiment = Experiment(experiment_name="exp1-chatqa-fsdp-large")

Task Training and Validation cancelled
The previously running experiment tensorboard-demo-1 was forcibly ended. Created a new experiment 'exp1-chatqa-fsdp-large' with Experiment ID: 3 and Metric Experiment ID: 3 at /home/rapid-fire/rapidfireai/rapidfire_experiments/exp1-chatqa-fsdp-large


### Define Custom Eval Metrics Function

In [5]:
def sample_compute_metrics(eval_preds):  
    """Optional function to compute eval metrics based on predictions and labels"""
    predictions, labels = eval_preds

    # Standard text-based eval metrics: Rouge and BLEU
    import evaluate
    rouge = evaluate.load("rouge")
    bleu = evaluate.load("bleu")

    rouge_output = rouge.compute(predictions=predictions, references=labels, use_stemmer=True)
    rouge_l = rouge_output["rougeL"]
    bleu_output = bleu.compute(predictions=predictions, references=labels)
    bleu_score = bleu_output["bleu"]

    return {
        "rougeL": round(rouge_l, 4),
        "bleu": round(bleu_score, 4),
    }

### Define Multi-Config Knobs for Model, LoRA, and SFT Trainer using RapidFire AI Wrapper APIs

In [6]:
# 2 LoRA PEFT configs lite with different adapter capacities
peft_configs_lite = List([
    RFLoraConfig(
        r=8,
        lora_alpha=16,
        lora_dropout=0.1,
        target_modules=["q_proj", "v_proj"],  # Standard transformer naming
        bias="none"
    ),
    RFLoraConfig(
        r=16,
        lora_alpha=32,
        lora_dropout=0.05,
        target_modules=["q_proj", "v_proj","k_proj"],  # Standard transformer naming
        bias="none"
    ),
])

#2 peft configs = 2 combinations in total
config_set_lite = List([
    RFModelConfig(
        model_name="meta-llama/Meta-Llama-3-70B-Instruct",  # 70B model
        peft_config=peft_configs_lite,
        training_args=RFSFTConfig(
            learning_rate=1e-4,  
            lr_scheduler_type="linear",
            per_device_train_batch_size=1,
            per_device_eval_batch_size=1,
            num_train_epochs=1,
            gradient_accumulation_steps=4,   
            logging_steps=1,
            eval_strategy="steps",
            eval_steps=3,
            gradient_checkpointing=True,
            gradient_checkpointing_kwargs={"use_reentrant": False},
            bf16=True,
            tf32=True,
            max_length=256,
            fsdp="full_shard auto_wrap",
            fsdp_config={"backward_prefetch": "backward_pre","forward_prefetch": False,"use_orig_params": False,  "cpu_ram_efficient_loading": True,"offload_params": False, "sync_module_states": True,"limit_all_gathers": True, "sharding_strategy": "FULL_SHARD",
                "auto_wrap_policy": "TRANSFORMER_BASED_WRAP"},            
        ),
        model_type="causal_lm",
        model_kwargs={"device_map":None, "torch_dtype": "bfloat16", "use_cache": False},
        formatting_func=sample_formatting_function,
        compute_metrics=sample_compute_metrics,
        generation_config = { # This is for text based evaluation/prediction for causal_lm models
            "max_new_tokens": 128,
            "do_sample": False,
            "use_cache": False,
        }
    ),
])


#### Define Model Creation Function for All Model Types Across Configs

In [7]:

def sample_create_model(model_config): 
     """Function to create model object for any given config; must return tuple of (model, tokenizer)"""
     from transformers import AutoModelForCausalLM, AutoTokenizer, AutoModelForSeq2SeqLM, AutoModelForMaskedLM, BitsAndBytesConfig
     import torch
     
     model_name = model_config["model_name"]
     model_type = model_config["model_type"]
     model_kwargs = model_config["model_kwargs"]

     bnb_config = BitsAndBytesConfig(
     load_in_4bit=True,
     bnb_4bit_use_double_quant=True,
     bnb_4bit_quant_type="nf4",
     bnb_4bit_compute_dtype=torch.bfloat16,
     bnb_4bit_quant_storage=torch.bfloat16
     )
     model_kwargs["quantization_config"] = bnb_config
 
     if model_type == "causal_lm":
          model = AutoModelForCausalLM.from_pretrained(model_name, **model_kwargs)
     elif model_type == "gpt":
          model = GptOssForCausalLM.from_pretrained(model_name, **model_kwargs)
     elif model_type == "seq2seq_lm":
          model = AutoModelForSeq2SeqLM.from_pretrained(model_name, **model_kwargs)
     elif model_type == "masked_lm":
          model = AutoModelForMaskedLM.from_pretrained(model_name, **model_kwargs)
     elif model_type == "custom":
          # Handle custom model loading logic, e.g., loading your own checkpoints
          # model = ... 
          pass
     else:
          # Default to causal LM
          model = AutoModelForCausalLM.from_pretrained(model_name, **model_kwargs)
      
     tokenizer = AutoTokenizer.from_pretrained(model_name)
     model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
     return (model,tokenizer)

#### Generate Config Group

In [8]:
# Simple grid search across all sets of config knob values = 2 combinations in total
config_group = RFGridSearch(
    configs=config_set_lite,
    trainer_type="SFT"
)

### Run Multi-Config Training

In [9]:
# Launch training of all configs in the config_group with swap granularity of 4 chunks
experiment.run_fit(
    config_group,
    sample_create_model,
    train_dataset,
    eval_dataset,
    num_chunks=4,
    seed=42,
    num_gpus=NUM_GPUS,
)

Started 1 worker processes successfully
Created workers
Run 1 has failed: You are trying to access a gated repo.
Make sure to have access to it at https://huggingface.co/meta-llama/Meta-Llama-3-70B-Instruct.
401 Client Error. (Request ID: Root=1-69dd077c-46fdf5075684c22329615e53;80bbfdde-db2e-473a-be02-dcfa0fb9598e)

Cannot access gated repo for url https://huggingface.co/meta-llama/Meta-Llama-3-70B-Instruct/resolve/main/config.json.
Access to model meta-llama/Meta-Llama-3-70B-Instruct is restricted. You must have access to it and be authenticated to access it. Please log in.Traceback (most recent call last):
  File "/home/rapid-fire/rapidfireai/venv/lib/python3.12/site-packages/huggingface_hub/utils/_http.py", line 403, in hf_raise_for_status
    response.raise_for_status()
  File "/home/rapid-fire/rapidfireai/venv/lib/python3.12/site-packages/requests/models.py", line 1028, in raise_for_status
    raise HTTPError(http_error_msg, response=self)
requests.exceptions.HTTPError: 401 Clien

### End Current Experiment

In [ ]:
experiment.end()